In [3]:
# =========================================================
# PySpark Text Processing and Word Frequency
# =========================================================

# ---------------------------------------------------------
# 0. Set Java 17 for PySpark
#    This must be done BEFORE creating SparkSession
# ---------------------------------------------------------

import os

os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
os.environ["PATH"] = (
    "/usr/lib/jvm/java-17-openjdk-amd64/bin:"
    + os.environ["PATH"]
)


# ---------------------------------------------------------
# 1. Import required PySpark libraries
# ---------------------------------------------------------

from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col,
    regexp_replace,
    lower,
    split,
    explode,
    trim
)


# ---------------------------------------------------------
# 2. Create Spark Session
# ---------------------------------------------------------

spark = SparkSession.builder \
    .appName("TextProcessingWordFrequency") \
    .master("local[*]") \
    .getOrCreate()

print("Spark version:", spark.version)
print("Spark started successfully!")


# ---------------------------------------------------------
# 3. Read the text file into a DataFrame
#    Requirement (a)
# ---------------------------------------------------------

df = spark.read.text("input.txt")

print("\n===== ORIGINAL DATA =====")
df.show(truncate=False)


# ---------------------------------------------------------
# 4. Check the schema
#    Requirement (b)
#
#    Each line becomes one record.
#    The column is named 'value' and is StringType.
# ---------------------------------------------------------

print("\n===== SCHEMA =====")
df.printSchema()


# ---------------------------------------------------------
# 5. Remove punctuation, quotation marks and symbols
#    Requirement (c)
#
#    [^a-zA-Z0-9\\s]
#    removes anything that is NOT:
#    - uppercase letters
#    - lowercase letters
#    - numbers
#    - whitespace
# ---------------------------------------------------------

df_clean = df.withColumn(
    "value",
    regexp_replace(
        col("value"),
        "[^a-zA-Z0-9\\s]",
        ""
    )
)

print("\n===== AFTER REMOVING PUNCTUATION AND SYMBOLS =====")
df_clean.show(truncate=False)


# ---------------------------------------------------------
# 6. Convert text to lowercase
#    Requirement (d)
# ---------------------------------------------------------

df_clean = df_clean.withColumn(
    "value",
    lower(col("value"))
)

print("\n===== AFTER CONVERTING TO LOWERCASE =====")
df_clean.show(truncate=False)


# ---------------------------------------------------------
# 7. Define stopwords
#    Requirement (d)
# ---------------------------------------------------------

stopwords = [
    "is",
    "the",
    "and",
    "or",
    "a",
    "an",
    "to",
    "of",
    "in"
]


# ---------------------------------------------------------
# 8. Split each line into words
#    Requirement (e)
# ---------------------------------------------------------

df_words = df_clean.withColumn(
    "words",
    split(trim(col("value")), "\\s+")
)

print("\n===== AFTER TOKENIZATION =====")
df_words.show(truncate=False)


# ---------------------------------------------------------
# 9. Create one row for each word
#    Requirement (e)
# ---------------------------------------------------------

df_exploded = df_words.select(
    explode(col("words")).alias("word")
)

print("\n===== INDIVIDUAL WORDS =====")
df_exploded.show()


# ---------------------------------------------------------
# 10. Remove stopwords and empty words
#     Requirement (d)
# ---------------------------------------------------------

df_filtered = df_exploded.filter(
    (col("word") != "") &
    (~col("word").isin(stopwords))
)

print("\n===== AFTER REMOVING STOPWORDS =====")
df_filtered.show()


# ---------------------------------------------------------
# 11. Count frequency of each word
#     Requirement (f)
# ---------------------------------------------------------

word_frequency = df_filtered.groupBy("word").count()


# ---------------------------------------------------------
# 12. Sort by frequency in descending order
# ---------------------------------------------------------

word_frequency = word_frequency.orderBy(
    col("count").desc()
)


# ---------------------------------------------------------
# 13. Display final word frequency
# ---------------------------------------------------------

print("\n===== FINAL WORD FREQUENCY =====")
word_frequency.show()


# ---------------------------------------------------------
# 14. Stop Spark Session
# ---------------------------------------------------------

spark.stop()

Spark version: 4.0.1
Spark started successfully!

===== ORIGINAL DATA =====
+---------------------------------------------------+
|value                                              |
+---------------------------------------------------+
|The quick brown fox jumps over the lazy dog.       |
|The fox is quick and the dog is lazy.              |
|Spark is a powerful tool for big data processing.  |
|Spark makes big data processing fast and efficient.|
|The quick brown fox is running in the forest.      |
+---------------------------------------------------+


===== SCHEMA =====
root
 |-- value: string (nullable = true)


===== AFTER REMOVING PUNCTUATION AND SYMBOLS =====
+--------------------------------------------------+
|value                                             |
+--------------------------------------------------+
|The quick brown fox jumps over the lazy dog       |
|The fox is quick and the dog is lazy              |
|Spark is a powerful tool for big data processing  |
|Spa

+--------------------------------------------------+
|value                                             |
+--------------------------------------------------+
|the quick brown fox jumps over the lazy dog       |
|the fox is quick and the dog is lazy              |
|spark is a powerful tool for big data processing  |
|spark makes big data processing fast and efficient|
|the quick brown fox is running in the forest      |
+--------------------------------------------------+


===== AFTER TOKENIZATION =====
+--------------------------------------------------+-----------------------------------------------------------+
|value                                             |words                                                      |
+--------------------------------------------------+-----------------------------------------------------------+
|the quick brown fox jumps over the lazy dog       |[the, quick, brown, fox, jumps, over, the, lazy, dog]      |
|the fox is quick and the dog is lazy 